In [1]:
# imports
import pandas as pd
import numpy as np
import xgboost as xgb
from xgboost import XGBClassifier
import shap

c:\Users\will6\miniconda3\envs\cs320\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
#### BASE

#tested various hyperparameters

#read csv
training = pd.read_csv("train_data_w.csv")

#features
features = ['seed_dif', 
            'avg_margin_dif',
            'avg_eff_A',
            'avg_opp_eff_A', 
            'avg_eff_B', 
            'avg_opp_eff_B',
            'avg_thr_per_A', 
            'ft_per_A', 
            'avg_fg_per_A', 
            'avg_fg_a_per_A',
            'avg_thr_a_per_A', 
            'avg_to_per_A', 
            'avg_blk_per_A',
            'avg_opp_fg_a_per_A', 
            'avg_opp_fg_per_A', 
            'avg_opp_to_per_A',
            'avg_or_per_A',
            'avg_foul_rec_per_A', 
            'avg_foul_per_A',
            'avg_thr_per_B', 
            'ft_per_B', 
            'avg_fg_per_B', 
            'avg_fg_a_per_B',
            'avg_thr_a_per_B', 
            'avg_to_per_B', 
            'avg_blk_per_B',
            'avg_opp_fg_a_per_B', 
            'avg_opp_fg_per_B', 
            'avg_opp_to_per_B',
            'avg_or_per_B',
            'avg_foul_rec_per_B', 
            'avg_foul_per_B',
            'thr_a_per_dif', 
            'tempo_pred', 
            'pred_fg_per_dif', 
            'pred_or_per_dif'
            ]


#Initialize arrays
training_dfs = []
base_val_dfs = []
val_years = []
years = [2011,2012,2013,2014,2015,2016,2017,2018,2019,2021,2022,2023,2024]

for i in years:
    if i == 2019:
        val_year = 2021
    else:
        val_year = i+1

    temp_training = training.query("Season <= @i")
    temp_val = training.query("Season == @val_year")

    training_dfs.append(temp_training)
    base_val_dfs.append(temp_val)
    val_years.append(val_year)

### Model

base_val_error = []

#model
base_mod = XGBClassifier(n_estimators=600, max_depth=2, learning_rate=0.01, subsample = 0.6, colsample_bynode = 0.8, min_split_loss = 6, max_bin = 20, min_child_weight = 2, num_parallel_tree = 1, objective='binary:logistic', seed = 323)

for i in range(len(years)):
    X_train = training_dfs[i][features]
    y_train = training_dfs[i]['result']

    val = base_val_dfs[i].copy()

    X_val = val[features]
    y_val = val['result']

    #fit model
    base_mod.fit(X_train, y_train)
    
    #make predictions
    preds = base_mod.predict_proba(X_val)
    val['pred'] = preds[:,1]
    val['loss'] = (val['pred'] - val['result'])**2

    base_val_dfs[i] = val.copy()
    print(val_years[i], "Loss:", np.mean(val['loss']))
    base_val_error.append(np.mean(val['loss']))

print("Last 5 Loss:", np.mean(base_val_error[-5:]))

#print(val_base.drop('loss', axis = 1).sort_values('pred', ascending = False).head(5))
#val_base.drop('loss', axis = 1).sort_values('pred', ascending = True).head(5)

#shap
explainer = shap.TreeExplainer(base_mod)
shap_values = explainer.shap_values(X_val)
#shap.summary_plot(shap_values, X_val, plot_type="bar", max_display=50)


2012 Loss: 0.1252367294825085
2013 Loss: 0.1634593735090288
2014 Loss: 0.14039321775698474
2015 Loss: 0.12279191141420114
2016 Loss: 0.1750416652835347
2017 Loss: 0.15185011710010235
2018 Loss: 0.16398571643455837
2019 Loss: 0.13240245756508365
2021 Loss: 0.14587989376998567
2022 Loss: 0.16095614968191932
2023 Loss: 0.16327790377231344
2024 Loss: 0.11983043829387308
2025 Loss: 0.11283157540700084
Last 5 Loss: 0.14055519218501847


In [ ]:
#Rolling

#read csv
training = pd.read_csv("train_data_w.csv")

#features
features = ['seed_dif', 
            'avg_margin_dif',
            'avg_eff_A',
            'avg_opp_eff_A', 
            'avg_eff_B', 
            'avg_opp_eff_B',
            'avg_thr_per_A', 
            'ft_per_A', 
            'avg_fg_per_A', 
            'avg_fg_a_per_A',
            'avg_thr_a_per_A', 
            'avg_to_per_A', 
            'avg_blk_per_A',
            'avg_opp_fg_a_per_A', 
            'avg_opp_fg_per_A', 
            'avg_opp_to_per_A',
            'avg_or_per_A',
            'avg_foul_rec_per_A', 
            'avg_foul_per_A',
            'avg_thr_per_B', 
            'ft_per_B', 
            'avg_fg_per_B', 
            'avg_fg_a_per_B',
            'avg_thr_a_per_B', 
            'avg_to_per_B', 
            'avg_blk_per_B',
            'avg_opp_fg_a_per_B', 
            'avg_opp_fg_per_B', 
            'avg_opp_to_per_B',
            'avg_or_per_B',
            'avg_foul_rec_per_B', 
            'avg_foul_per_B',
            'thr_a_per_dif', 
            'tempo_pred', 
            'pred_fg_per_dif', 
            'pred_or_per_dif'
            ]


#Initialize arrays
training_dfs = []
r_val_dfs = []
val_years = []
years = [2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2021,2022,2023,2024]

y_count = 8

for i in years:
    if i == 2019:
        val_year = 2021
    else:
        val_year = i+1

    if i <= 2019 or i >= 2020 + y_count:
        seasons = [i - k for k in range(y_count)]
    else:
        seasons = [i - k for k in range(y_count+1)]

    temp_training = training.query("Season in @seasons")
    temp_val = training.query("Season == @val_year")

    training_dfs.append(temp_training)
    r_val_dfs.append(temp_val)
    val_years.append(val_year)

### Model

#model
r_mod = XGBClassifier(n_estimators=500, max_depth=2, learning_rate=0.015, subsample = 0.6, colsample_bynode = 0.8, min_split_loss = 4, max_bin = 20, min_child_weight = 4, num_parallel_tree = 1, objective='binary:logistic', seed = 323)

r_val_error = []

for i in range(len(years)):
    X_train = training_dfs[i][features]
    y_train = training_dfs[i]['result']

    val = r_val_dfs[i].copy()

    X_val = val[features]
    y_val = val['result']

    #fit model
    r_mod.fit(X_train, y_train)
    
    #make predictions
    preds = r_mod.predict_proba(X_val)
    val['pred'] = preds[:,1]
    val['loss'] = (val['pred'] - val['result'])**2

    r_val_dfs[i] = val.copy()
    r_val_error.append(np.mean(val['loss']))
    print(val_years[i], "Loss:", np.mean(val['loss']), " |  Dif:", np.mean(val['loss']) - base_val_error[val_years[i] - 2017])

print("Last 5 Loss:", np.mean(r_val_error[-5:]), " | Dif:", np.mean(r_val_error[-5:]) - np.mean(base_val_error[-5:]))

#shap
explainer = shap.TreeExplainer(r_mod)
shap_values = explainer.shap_values(X_val)
#shap.summary_plot(shap_values, X_val, plot_type="bar", max_display=50)


2011 Loss: 0.14659917305421122  |  Dif: 0.01419671548912757
2012 Loss: 0.1261860376637552  |  Dif: -0.01969385610623048
2013 Loss: 0.17066094866217368  |  Dif: 0.009704798980254364
2014 Loss: 0.14145280883284878  |  Dif: -0.021825094939464662
2015 Loss: 0.11498867537984972  |  Dif: -0.0048417629140233565
2016 Loss: 0.18514475270191766  |  Dif: 0.07231317729491682
2017 Loss: 0.15775391941145414  |  Dif: 0.032517189928945645
2018 Loss: 0.16367915432274  |  Dif: 0.00021978081371118452
2019 Loss: 0.13697482699026528  |  Dif: -0.003418390766719459
2021 Loss: 0.14494559149379466  |  Dif: -0.030096073789740052
2022 Loss: 0.16215313230418646  |  Dif: 0.01030301520408411
2023 Loss: 0.16830598632698174  |  Dif: 0.004320269892423373
2024 Loss: 0.13182826282814059  |  Dif: -0.0005741947369430622
2025 Loss: 0.10591788821057302  |  Dif: -0.03996200555941265
Last 5 Loss: 0.14263017223273528  | Dif: 0.0020749800477168068


In [ ]:
#### Weights

#read csv
training = pd.read_csv("train_data_w.csv")

#features
features = ['seed_dif', 
            'avg_margin_dif',
            'avg_eff_A',
            'avg_opp_eff_A', 
            'avg_eff_B', 
            'avg_opp_eff_B',
            'avg_thr_per_A', 
            'ft_per_A', 
            'avg_fg_per_A', 
            'avg_fg_a_per_A',
            'avg_thr_a_per_A', 
            'avg_to_per_A', 
            'avg_blk_per_A',
            'avg_opp_fg_a_per_A', 
            'avg_opp_fg_per_A', 
            'avg_opp_to_per_A',
            'avg_or_per_A',
            'avg_foul_rec_per_A', 
            'avg_foul_per_A',
            'avg_thr_per_B', 
            'ft_per_B', 
            'avg_fg_per_B', 
            'avg_fg_a_per_B',
            'avg_thr_a_per_B', 
            'avg_to_per_B', 
            'avg_blk_per_B',
            'avg_opp_fg_a_per_B', 
            'avg_opp_fg_per_B', 
            'avg_opp_to_per_B',
            'avg_or_per_B',
            'avg_foul_rec_per_B', 
            'avg_foul_per_B',
            'thr_a_per_dif', 
            'tempo_pred', 
            'pred_fg_per_dif', 
            'pred_or_per_dif'
            ]

#Initialize arrays
training_dfs = []
w_val_dfs = []
val_years = []
years = [2011,2012,2013,2014,2015,2016,2017,2018,2019,2021,2022,2023,2024]

#Weight parameter
weight_param = 0.95

for i in years:
    if i == 2019:
        val_year = 2021
    else:
        val_year = i+1

    temp_training = training.query("Season <= @i")
    temp_val = training.query("Season == @val_year")

    training_dfs.append(temp_training)
    w_val_dfs.append(temp_val)
    val_years.append(val_year)

### Model

#model
w_mod = XGBClassifier(n_estimators=500, max_depth=2, learning_rate=0.015, subsample = 0.6, colsample_bynode = 0.8, min_split_loss = 6, max_bin = 20, min_child_weight = 2, num_parallel_tree = 1, objective='binary:logistic', seed = 323)

w_val_error = []

for i in range(len(years)):
    temp_df = training_dfs[i]
    temp_df = temp_df.assign(weight = (weight_param ** (temp_df['Season'].max() - temp_df['Season'])).clip(0.1))

    X_train = temp_df[features]
    y_train = temp_df['result']
    train_weights = temp_df['weight']

    val = w_val_dfs[i].copy()

    X_val = val[features]
    y_val = val['result']

    #fit model
    w_mod.fit(X_train, y_train, sample_weight=train_weights)
    
    #make predictions
    preds = w_mod.predict_proba(X_val)
    val['pred'] = preds[:,1]
    val['loss'] = (val['pred'] - val['result'])**2

    w_val_dfs[i] = val.copy()
    w_val_error.append(np.mean(val['loss']))
    print(val_years[i], "Loss:", np.mean(val['loss']), " |  Dif:", np.mean(val['loss']) - base_val_error[val_years[i] - 2017])

print("Last 5 Loss:", np.mean(w_val_error[-5:]), " | Dif:", np.mean(w_val_error[-5:]) - np.mean(base_val_error[-5:]))

#shap
explainer = shap.TreeExplainer(w_mod)
shap_values = explainer.shap_values(X_val)
#shap.summary_plot(shap_values, X_val, plot_type="bar", max_display=50)


2012 Loss: 0.1246030127863913  |  Dif: -0.02127688098359437
2013 Loss: 0.16368688505183518  |  Dif: 0.00273073536991586
2014 Loss: 0.14179024692480277  |  Dif: -0.021487656847510667
2015 Loss: 0.12175425902218485  |  Dif: 0.0019238207283117786
2016 Loss: 0.1792142098287499  |  Dif: 0.06638263442174905
2017 Loss: 0.15360575505184307  |  Dif: 0.02836902556933457
2018 Loss: 0.1629469789254199  |  Dif: -0.0005123945836089105
2019 Loss: 0.13081875916014382  |  Dif: -0.009574458596840918
2021 Loss: 0.14410706407351698  |  Dif: -0.03093460121001773
2022 Loss: 0.16240093519469623  |  Dif: 0.010550818094593878
2023 Loss: 0.16398812162389867  |  Dif: 2.4051893403043145e-06
2024 Loss: 0.12466606823920857  |  Dif: -0.007736389325875073
2025 Loss: 0.11400700289687397  |  Dif: -0.031872890873111695
Last 5 Loss: 0.14183383840563887  | Dif: 0.0012786462206204052


In [ ]:
### Combine

#Merge Data
def get_vals(lst, prefix):
    df = pd.concat(lst, ignore_index=True)[['Season', 'DayNum', 'team_A', 'team_B', 'score_A', 'score_B', 'result', 'pred', 'loss']]
    df = df.rename({'pred': f'{prefix}pred', 'loss': f'{prefix}loss'}, axis='columns')
    return df

base_val = get_vals(base_val_dfs, "base_")
r_val = get_vals(r_val_dfs, "r_")
w_val = get_vals(w_val_dfs, "w_")

merge_keys = ["Season", 'DayNum', 'team_A', 'team_B', 'score_A', 'score_B', 'result']

full_val = pd.merge(base_val, r_val, how="outer", on=merge_keys).merge(
        w_val, how="outer", on=merge_keys)


full_val.tail()


#combined prediction
base_weight = 0.4
r_weight = 0.3
w_weight = 0.3

full_val = full_val.assign(com_pred = full_val['base_pred']*base_weight + full_val['r_pred']*r_weight + full_val['w_pred']*w_weight)
full_val['com_loss'] = (full_val['com_pred'] - full_val['result'])**2

full_val.tail()

#group and summarize
val_summary = full_val.groupby(["Season"]).agg(
    base_val=("base_loss", "mean"),
    r_val=("r_loss", "mean"),
    w_val=("w_loss", "mean"),
    com_val=("com_loss", "mean")).reset_index()

val_summary.loc['last5'] = val_summary.set_index('Season').iloc[-5:].mean()

val_summary.tail(6)

,Season,base_val,r_val,w_val,com_val
9,2021.0,0.145880,0.144946,0.144107,0.144672
10,2022.0,0.160956,0.162153,0.162401,0.161397
11,2023.0,0.163278,0.168306,0.163988,0.164458
12,2024.0,0.119830,0.131828,0.124666,0.124427
13,2025.0,0.112832,0.105918,0.114007,0.110507
last5,NaN,0.140555,0.142630,0.141834,0.141092
